In [1]:
print("hello world")

hello world


In [2]:
import requests
import pandas as pd

API_KEY = "96c6a7e11526aa348124e9e23aa001a5"  # 일단 테스트용

url = "http://www.kobis.or.kr/kobisopenapi/webservice/rest/boxoffice/searchDailyBoxOfficeList.json"

params = {
    "key": API_KEY,
    "targetDt": "20240101"
}

res = requests.get(url, params=params)
data = res.json()

df = pd.DataFrame(data['boxOfficeResult']['dailyBoxOfficeList'])

df.head()

,rnum,rank,rankInten,rankOldAndNew,movieCd,movieNm,openDt,salesAmt,salesShare,salesInten,salesChange,salesAcc,audiCnt,audiInten,audiChange,audiAcc,scrnCnt,showCnt
0,1,1,0,OLD,20203702,노량: 죽음의 바다,2023-12-20,2893509165,38.8,-686119312,-19.2,36926118105,290717,-59184,-16.9,3728850,1706,5793
1,2,2,0,OLD,20212866,서울의 봄,2023-11-22,2666429137,35.8,-649456909,-19.6,118016920547,262422,-64054,-19.6,12117201,1614,5444
2,3,3,0,OLD,20236146,신차원! 짱구는 못말려 더 무비 초능력 대결전 ~날아라 수제김밥~,2023-12-22,565417320,7.6,-69888800,-11,6071745009,58239,-6751,-10.4,626508,834,1671
3,4,4,0,OLD,20235735,아쿠아맨과 로스트 킹덤,2023-12-20,507984981,6.8,-156026099,-23.5,7967584753,49368,-14686,-22.9,772354,832,1611
4,5,5,0,OLD,20235596,트롤: 밴드 투게더,2023-12-20,273538754,3.7,-16945147,-5.8,3383181955,28988,-1952,-6.3,365927,660,981


In [3]:
from datetime import datetime, timedelta
import requests
import pandas as pd
import os

API_KEY = "96c6a7e11526aa348124e9e23aa001a5"
url = "http://www.kobis.or.kr/kobisopenapi/webservice/rest/boxoffice/searchDailyBoxOfficeList.json"

start = datetime(2001, 1, 1)
end = datetime(2025, 12, 31)

all_data = []
failed_dates = []

current = start

session = requests.Session()

while current <= end:
    date_str = current.strftime("%Y%m%d")

    params = {
        "key": API_KEY,
        "targetDt": date_str
    }

    try:
        res = session.get(url, params=params, timeout=10)
        res.raise_for_status()
        data = res.json()

        daily = data.get("boxOfficeResult", {}).get("dailyBoxOfficeList", [])

        for row in daily:
            row["date"] = date_str

        all_data.extend(daily)

    except Exception as e:
        failed_dates.append((date_str, str(e)))

    current += timedelta(days=1)

df = pd.DataFrame(all_data)

file_name = "kobis_boxoffice_2001_2025.csv"
df.to_csv(file_name, index=False, encoding="utf-8-sig")

size_mb = os.path.getsize(file_name) / (1024 * 1024)
print("총 행 개수:", df.shape)
print("수집 실패 날짜 수:", len(failed_dates))
print(f"CSV 크기: {size_mb:.2f} MB")

총 행 개수: (19429, 19)
수집 실패 날짜 수: 0
CSV 크기: 2.40 MB


In [4]:
test_dates = ["20010101", "20050101", "20100101", "20150101", "20200101", "20250101"]

for d in test_dates:
    params = {"key": API_KEY, "targetDt": d}
    res = requests.get(url, params=params)
    data = res.json()
    print("날짜:", d)
    print(data)
    print("=" * 80)

날짜: 20010101
{'faultInfo': {'message': '키의 하루 이용량을 초과하였습니다.', 'errorCode': '320011'}}
날짜: 20050101
{'faultInfo': {'message': '키의 하루 이용량을 초과하였습니다.', 'errorCode': '320011'}}
날짜: 20100101
{'faultInfo': {'message': '키의 하루 이용량을 초과하였습니다.', 'errorCode': '320011'}}
날짜: 20150101
{'faultInfo': {'message': '키의 하루 이용량을 초과하였습니다.', 'errorCode': '320011'}}
날짜: 20200101
{'faultInfo': {'message': '키의 하루 이용량을 초과하였습니다.', 'errorCode': '320011'}}
날짜: 20250101
{'faultInfo': {'message': '키의 하루 이용량을 초과하였습니다.', 'errorCode': '320011'}}


In [5]:
import pandas as pd

df = pd.read_csv("kobis_boxoffice_2001_2025.csv")
print(df["date"].min())
print(df["date"].max())
print(df["date"].nunique())

20030112
20090318
1955


In [6]:
date_counts = df.groupby("date").size().reset_index(name="cnt")
print(date_counts.head())
print(date_counts.tail(20))

       date  cnt
0  20030112    1
1  20031111    7
2  20031112    8
3  20031113    8
4  20031114    8
          date  cnt
1935  20090227   10
1936  20090228   10
1937  20090301   10
1938  20090302   10
1939  20090303   10
1940  20090304   10
1941  20090305   10
1942  20090306   10
1943  20090307   10
1944  20090308   10
1945  20090309   10
1946  20090310   10
1947  20090311   10
1948  20090312   10
1949  20090313   10
1950  20090314   10
1951  20090315   10
1952  20090316   10
1953  20090317   10
1954  20090318   10


In [7]:
import pandas as pd

# 파일명은 네 실제 파일명으로 바꿔
file1 = "kobis_boxoffice_2001_2025.csv"      # 2009-03-18까지 모인 파일
file2 = "test_kobis.csv"      # 예전에 만든 2015~2025 파일

def check_boxoffice_csv(file_name):
    df = pd.read_csv(file_name)

    # date를 문자열 8자리로 통일
    df["date"] = df["date"].astype(str).str.zfill(8)

    print("=" * 70)
    print("파일명:", file_name)
    print("행 개수:", df.shape)
    print("최소 날짜:", df["date"].min())
    print("최대 날짜:", df["date"].max())
    print("고유 날짜 수:", df["date"].nunique())

    date_counts = df.groupby("date").size().reset_index(name="cnt")
    print("\n앞 5개 날짜")
    print(date_counts.head())
    print("\n뒤 5개 날짜")
    print(date_counts.tail())

    return df, date_counts

df1, date_counts1 = check_boxoffice_csv(file1)
df2, date_counts2 = check_boxoffice_csv(file2)

파일명: kobis_boxoffice_2001_2025.csv
행 개수: (19429, 19)
최소 날짜: 20030112
최대 날짜: 20090318
고유 날짜 수: 1955

앞 5개 날짜
       date  cnt
0  20030112    1
1  20031111    7
2  20031112    8
3  20031113    8
4  20031114    8

뒤 5개 날짜
          date  cnt
1950  20090314   10
1951  20090315   10
1952  20090316   10
1953  20090317   10
1954  20090318   10
파일명: test_kobis.csv
행 개수: (29990, 19)
최소 날짜: 20150101
최대 날짜: 20230318
고유 날짜 수: 2999

앞 5개 날짜
       date  cnt
0  20150101   10
1  20150102   10
2  20150103   10
3  20150104   10
4  20150105   10

뒤 5개 날짜
          date  cnt
2994  20230314   10
2995  20230315   10
2996  20230316   10
2997  20230317   10
2998  20230318   10


In [8]:
def find_missing_dates(df, start_date, end_date):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")

    full_range = pd.date_range(start=start_date, end=end_date, freq="D")
    existing_dates = pd.Series(df["date"].unique())

    missing = sorted(set(full_range) - set(existing_dates))
    print(f"누락 날짜 수: {len(missing)}")

    if missing:
        print("누락 날짜 예시 20개:")
        for d in missing[:20]:
            print(d.strftime("%Y%m%d"))

    return missing

# 예시
missing1 = find_missing_dates(df1, "2001-01-01", "2025-12-31")
missing2 = find_missing_dates(df2, "2015-01-01", "2025-12-31")

누락 날짜 수: 7176
누락 날짜 예시 20개:
20010101
20010102
20010103
20010104
20010105
20010106
20010107
20010108
20010109
20010110
20010111
20010112
20010113
20010114
20010115
20010116
20010117
20010118
20010119
20010120
누락 날짜 수: 1019
누락 날짜 예시 20개:
20230319
20230320
20230321
20230322
20230323
20230324
20230325
20230326
20230327
20230328
20230329
20230330
20230331
20230401
20230402
20230403
20230404
20230405
20230406
20230407


In [9]:
df_all = pd.concat([df1, df2], ignore_index=True)
df_all["date"] = df_all["date"].astype(str).str.zfill(8)

print("=" * 70)
print("합친 파일 기준")
print("행 개수:", df_all.shape)
print("최소 날짜:", df_all["date"].min())
print("최대 날짜:", df_all["date"].max())
print("고유 날짜 수:", df_all["date"].nunique())

date_counts_all = df_all.groupby("date").size().reset_index(name="cnt")
print("\n뒤 10개 날짜")
print(date_counts_all.tail(10))

합친 파일 기준
행 개수: (49419, 19)
최소 날짜: 20030112
최대 날짜: 20230318
고유 날짜 수: 4954

뒤 10개 날짜
          date  cnt
4944  20230309   10
4945  20230310   10
4946  20230311   10
4947  20230312   10
4948  20230313   10
4949  20230314   10
4950  20230315   10
4951  20230316   10
4952  20230317   10
4953  20230318   10


In [2]:
from datetime import datetime, timedelta
import requests
import pandas as pd
import os

API_KEY = "96c6a7e11526aa348124e9e23aa001a5"
url = "http://www.kobis.or.kr/kobisopenapi/webservice/rest/boxoffice/searchDailyBoxOfficeList.json"

# 수집 구간 (2009-03-19 ~ 2014-12-31)
start = datetime(2009, 3, 19)
end = datetime(2014, 12, 31)

all_data = []
failed_dates = []

current = start
session = requests.Session()

while current <= end:
    date_str = current.strftime("%Y%m%d")

    params = {
        "key": API_KEY,
        "targetDt": date_str
    }

    try:
        res = session.get(url, params=params, timeout=10)
        res.raise_for_status()
        data = res.json()

        # API 제한 감지 (핵심 추가)
        if "faultInfo" in data:
            print(f"중단됨: {date_str} | {data['faultInfo']}")
            break

        # 정상 구조 체크
        if "boxOfficeResult" not in data:
            failed_dates.append((date_str, data))
            current += timedelta(days=1)
            continue

        daily = data["boxOfficeResult"].get("dailyBoxOfficeList", [])

        for row in daily:
            row["date"] = date_str

        all_data.extend(daily)

    except Exception as e:
        failed_dates.append((date_str, str(e)))

    # 진행상황 로그
    if date_str.endswith("0101"):
        print(f"진행 중: {date_str} | 누적 행 수: {len(all_data)}")

    current += timedelta(days=1)

# 저장
df = pd.DataFrame(all_data)

file_name = "kobis_boxoffice_20090319_20141231.csv"
df.to_csv(file_name, index=False, encoding="utf-8-sig")

# 결과 확인
size_mb = os.path.getsize(file_name) / (1024 * 1024)
print("="*60)
print("수집 완료")
print("행 개수:", df.shape)
print("실패 날짜 수:", len(failed_dates))
print(f"CSV 크기: {size_mb:.2f} MB")

진행 중: 20100101 | 누적 행 수: 2890
진행 중: 20110101 | 누적 행 수: 6540
진행 중: 20120101 | 누적 행 수: 10160
진행 중: 20130101 | 누적 행 수: 13820
진행 중: 20140101 | 누적 행 수: 17470
수집 완료
행 개수: (21110, 19)
실패 날짜 수: 3
CSV 크기: 2.66 MB


In [3]:
import pandas as pd
import os

# 파일 로드
df1 = pd.read_csv("kobis_boxoffice_2001_2025.csv")
df2 = pd.read_csv("kobis_boxoffice_20090319_20141231.csv")
df3 = pd.read_csv("test_kobis.csv")

# date 컬럼 통일 (문자열 8자리)
for df in [df1, df2, df3]:
    df["date"] = df["date"].astype(str).str.zfill(8)

# 병합
df_all = pd.concat([df1, df2, df3], ignore_index=True)

print("합치기 전 행 개수:", df_all.shape)

# 완전 중복 제거
df_all = df_all.drop_duplicates()

print("중복 제거 후 행 개수:", df_all.shape)

# 날짜 기준 정렬
df_all = df_all.sort_values("date")

# 저장
file_name = "kobis_boxoffice_1st_full.csv"
df_all.to_csv(file_name, index=False, encoding="utf-8-sig")

# 파일 크기 확인
size_mb = os.path.getsize(file_name) / (1024 * 1024)

print("="*60)
print("1차 통합 완료")
print("최종 행 개수:", df_all.shape)
print(f"CSV 크기: {size_mb:.2f} MB")
print("날짜 범위:", df_all["date"].min(), "~", df_all["date"].max())

합치기 전 행 개수: (70529, 19)
중복 제거 후 행 개수: (70529, 19)
1차 통합 완료
최종 행 개수: (70529, 19)
CSV 크기: 8.86 MB
날짜 범위: 20030112 ~ 20230318


In [4]:
df_all["date"] = pd.to_datetime(df_all["date"], format="%Y%m%d")

full_range = pd.date_range(
    start=df_all["date"].min(),
    end=df_all["date"].max(),
    freq="D"
)

missing = sorted(set(full_range) - set(df_all["date"].unique()))

print("누락 날짜 수:", len(missing))

누락 날짜 수: 306


In [5]:
missing_dates = sorted(missing)

# 문자열로 변환
missing_str = [d.strftime("%Y%m%d") for d in missing_dates]

# 연속 구간 찾기
gaps = []
start = missing_dates[0]
prev = missing_dates[0]

for d in missing_dates[1:]:
    if (d - prev).days == 1:
        prev = d
    else:
        gaps.append((start, prev))
        start = d
        prev = d

gaps.append((start, prev))

# 출력
print("누락 구간 개수:", len(gaps))

print("\n상위 10개 누락 구간:")
for s, e in gaps[:10]:
    print(s.strftime("%Y%m%d"), "~", e.strftime("%Y%m%d"), 
          f"({(e - s).days + 1}일)")

누락 구간 개수: 5

상위 10개 누락 구간:
20030113 ~ 20031110 (302일)
20031118 ~ 20031118 (1일)
20110810 ~ 20110810 (1일)
20110812 ~ 20110812 (1일)
20110815 ~ 20110815 (1일)
